创建一个人工数据集，并存储在csv（逗号分隔值）文件

In [1]:
# 导入 os 模块，用来操作文件夹和文件路径
import os
import pandas as pd

# 创建一个名为 data 的文件夹，位置在上一级目录（.. 表示上一级）
# exist_ok=True 意思是：如果这个文件夹已经存在，就不要报错，直接忽略
os.makedirs(os.path.join('..', 'data'), exist_ok=True)

# 组合出文件路径：上一级目录/data/house_tiny.csv
data_file = os.path.join('..', 'data', 'house_tiny.csv')

# 打开这个文件，以写入模式（'w'），并起个别名叫 f
# with 语句可以保证文件使用完后自动关闭，不用手动 close
with open(data_file, 'w') as f:
    # 写入表头：三个列名分别是 NumRooms、Alley、Price，结尾 \n 表示换行
    f.write("NumRooms,Alley,Price\n")
    # 写入第1行数据：NumRooms 缺失（用 NA 表示），Alley 是 Pave，价格 127500
    f.write('NA,Pave,127500\n')
    # 写入第2行数据：房间数 2，Alley 缺失，价格 106000
    f.write('2,NA,106000\n')
    # 写入第3行数据：房间数 4，Alley 缺失，价格 178100
    f.write('4,NA,178100\n')
    # 写入第4行数据：房间数缺失，Alley 缺失，价格 140000
    f.write('NA,NA,140000\n')
    
data=pd.read_csv(data_file)
print(data)
    


   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


为了处理缺失的数据，典型的方法包括插值和删除，这里我们考虑插值。

In [2]:
# 假设 data 已经读取好，是包含三列的 DataFrame
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2:]

# 只对数值类型的列填充平均值，避免对字符串列（如 Alley）求平均导致 TypeError
numeric_cols = inputs.select_dtypes(include='number').columns # 这都是数字
inputs[numeric_cols] = inputs[numeric_cols].fillna(inputs[numeric_cols].mean()) # 对NaN类型进行处理

print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于input中的类别或者零散值，我们将“NaN”视作一个类别。

In [3]:
# 把 inputs 表格中的文字列（比如 Alley 列，里面是 'Pave' 或 NaN）转换成“哑变量”列
# 哑变量就是为每个不同的类别建一列，用 1/0 表示“是不是这个类别”
# dummy_na=True 表示：如果原数据里有缺失值（NaN），也要为“缺失”单独建一列
inputs = pd.get_dummies(inputs, dummy_na=True) # 真取值为1，假取值为0

# 打印转换后的 inputs 看看效果
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


In [4]:
import torch

x,y=torch.tensor(inputs.to_numpy(dtype=float)),torch.tensor(outputs.to_numpy(dtype=float))
print(x,y)

tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]], dtype=torch.float64) tensor([[127500.],
        [106000.],
        [178100.],
        [140000.]], dtype=torch.float64)
